# Dynamic Bayesian Network — Machine-Failure Prediction from Alarm Data

I build a Dynamic Bayesian Network that predicts if a
machine will transition to a **Failure** state in the next time window, given its
current state and the alarm behaviour observed in the current window.

$$P(\text{State}_{t+1} = \text{Failure} \mid \text{State}_t,\ \text{alarm features at } t)$$

### 1. What is a Bayesian Network (BN)?
A Bayesian Network is a probabilistic graphical model that represents a set of random variables and their conditional dependencies through a Directed Acyclic Graph (DAG). Grounded in Bayes' Theorem, it lets to compactly represent the joint probability distribution of an entire system by exploiting the conditional independencies between variables. Bayesian Networks are widely used for diagnostic and predictive reasoning under uncertainty.

### 2. What is a Dynamic Bayesian Network (DBN)?
A Dynamic Bayesian Network extends the traditional Bayesian Network to model temporal, sequential, or time-series data. While a standard BN captures a static "snapshot" of a system, a DBN connects multiple BNs sequentially across discrete time slices. It assumes the Markov property, the state of the system at time $t$ depends only on the state at time $t-1$, which keeps the model computationally tractable.

### 3. How are temporal dependencies represented in a DBN?
Temporal dependencies are modelled with **transition edges** (also called inter-slice edges) that cross from one time slice to the next. For example, a directed edge connects a node at time $t$ (e.g. `Machine_State_t`) to a node at time $t+1$ (`Machine_State_t+1`). These cross-slice arrows explicitly capture how past observations and history influence the future state.

### 4. What are Nodes, Directed Edges, and Conditional Probability Tables (CPTs)?
- **Nodes:** the building blocks of the graph, each representing a random variable. In this exercise the nodes represent variables such as *Alarm Count*, *Alarm Duration*, and the *Machine State* (Running / Failure).
- **Directed Edges:** arrows connecting nodes to describe influence or direct conditional dependency. An arrow from node $A$ to node $B$ means that $B$ is probabilistically conditioned on $A$.
- **Conditional Probability Tables (CPTs):** tables attached to each node that quantify the probability distribution of that node given every possible combination of its parents' states.

### Imports and Setup

In [ ]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore")

from pgmpy.models import DynamicBayesianNetwork as DBN

# Alarm feature columns
FEATURES = ['A1_Count', 'A1_Duration', 'A2_Count', 'A2_Duration', 'A3_Count', 'A3_Duration']

print("Libraries imported successfully!")

### Calulation of the alarm durations

In [ ]:
df=pd.read_csv('dataset_exercise.csv', delimiter=';')

#Convert to datetime
df['start_alarm']=pd.to_datetime(df['start_alarm'])
df['end_alarm']=pd.to_datetime(df['end_alarm'])

#Duration of the alarms
df['duration_seconds']= (df['end_alarm']-df['start_alarm']).dt.total_seconds()
df.head()


### Functions based on Step1 for alarm count and duration

In [ ]:
def alarm_count_type(count):
    if count == 0: return 'None'
    elif 1 <= count <= 2: return 'Low'
    elif 3 <= count <= 5: return 'Medium'
    else: return 'High'

def alarm_duration_type(duration):
    if duration == 0: return 'None'
    elif 1 <= duration <= 10: return 'Short'
    elif 11 <= duration <= 60: return 'Medium'
    else: return 'Long'

### Get the alarms which are the most frequent

In [ ]:
top_alarms=df['alarm_id'].value_counts().head(3).index.tolist()
for i, aid in enumerate(top_alarms, 1):
    print(f"A{i} = {aid}  ({(df['alarm_id'] == aid).sum()} occurrences)")

### Function for Step 2 builing a feature row per time windows
Compute each alarm's cound and duration, converting them to the categories in Step1 and label them with the major machine state in the window

In [ ]:
def build_window_features(df, alarm_ids):
    processed_data = []
    for w, window_data in df.groupby('time_window'):
        state = window_data['machine_state'].mode().iloc[0]   # majority label
        row   = {'time_window': w, 'State': state}
        for i, alarm in enumerate(top_alarms, start=1):
            sub = window_data[window_data['alarm_id'] == alarm]
            row[f'A{i}_Count']    = alarm_count_type(len(sub))
            row[f'A{i}_Duration'] = alarm_duration_type(sub['duration_seconds'].sum())
        processed_data.append(row)
    return (pd.DataFrame(processed_data)
              .sort_values('time_window')
              .reset_index(drop=True))

### Apply the function

In [ ]:
df_processed = build_window_features(df, top_alarms)

print(f"{len(df_processed)} windows built")
print("State distribution:", df_processed['State'].value_counts().to_dict())
df_processed.head()

### Step3 Generate the training transition
The DBN learns from pairs of consecutive windows: the time t paired with t+1
I use `shift(-1)` to pair each window with the next, and filter out gaps because there are gaps in the `df['time_window']` column

In [ ]:
df_processed['State_next']   = df_processed['State'].shift(-1)
df_processed['next_window']  = df_processed['time_window'].shift(-1)

#Check if next window is consecutive
df_transitions=df_processed[df_processed['next_window']==df_processed['time_window']+1].dropna().copy()

print(f"{len(df_transitions)} valid consecutive transitions")
print("Next-state distribution:", df_transitions['State_next'].value_counts().to_dict())
df_transitions.head()

### Step4 Constructing the DBN
I declare the temporal dependencies using pgmpy's `DynamicBayesianNetwork`.
Nodes are represented as tuples `(variable_name, time_slice)`:
- `('State', 0)` = current state
- `('State', 1)` = next state

Every current feature and the current state point into the next state:

```
('State', 0)       ──►  ('State', 1)
('A1_Count', 0)    ──►  ('State', 1)
('A1_Duration', 0) ──►  ('State', 1)
('A2_Count', 0)    ──►  ('State', 1)
('A2_Duration', 0) ──►  ('State', 1)
('A3_Count', 0)    ──►  ('State', 1)
('A3_Duration', 0) ──►  ('State', 1)
```

In [ ]:
dbn = DBN()

edges = [
    (('State', 0),       ('State', 1)),
    (('A1_Count', 0),    ('State', 1)),
    (('A1_Duration', 0), ('State', 1)),
    (('A2_Count', 0),    ('State', 1)),
    (('A2_Duration', 0), ('State', 1)),
    (('A3_Count', 0),    ('State', 1)),
    (('A3_Duration', 0), ('State', 1)),
]

#Prevent inference from crashing
alarm_features = ['A1_Count', 'A1_Duration', 'A2_Count', 'A2_Duration', 'A3_Count', 'A3_Duration']
for feature in alarm_features:
    edges.append(((feature, 0), (feature, 1)))

dbn.add_edges_from(edges)

print("DBN structure defined.")

### Step 5: Format the Data and Train the DBN

`pgmpy` requires the training data columns to exactly match the `(Variable, Time_Slice)` tuple format defined in the network edges.

I will map the columns from the `df_transitions` dataframe into a new `dbn_data` dataframe. When the data is properly formatted, I will use Maximum Likelihood Estimation to train the network and learn the Conditional Probability Tables (CPTs).

In [ ]:
dbn_data = pd.DataFrame()

# 1. Map the Current Window features
dbn_data[('State', 0)] = df_transitions['State']
for feature in alarm_features:
    dbn_data[(feature, 0)] = df_transitions[feature]

# 2. Map the Future Window target
dbn_data[('State', 1)] = df_transitions['State_next']

#Provide dummy data for Time 1 alarms so the MLE can learn their CPDs
for feature in alarm_features:
    dbn_data[(feature, 1)] = df_transitions[feature]

dbn.fit(dbn_data, estimator='MLE')

print("Model trained successfully! All nodes are present.")

### Expected Outcome/Inference
Use the example in the instructions to test the network

In [ ]:
from pgmpy.inference import DBNInference

dbn_infer = DBNInference(dbn)

evidence = {
    ('State', 0): 'Running',
    ('A1_Count', 0): 'High',
    ('A1_Duration', 0): 'Long',
    ('A2_Count', 0): 'Medium',
    ('A2_Duration', 0): 'Medium',
    ('A3_Count', 0): 'Low',
    ('A3_Duration', 0): 'Short'
}

print("Executing query: P( State_1 | Evidence )\n")

try:
    result = dbn_infer.query(variables=[('State', 1)], evidence=evidence)

    # Extract the factor from the returned pgmpy dictionary
    factor = result[('State', 1)]

    # Map the state names to their probability values
    prob_dict = dict(zip(factor.state_names[('State', 1)], factor.values))

    print("--- Predicted Probabilities ---")
    for state, prob in prob_dict.items():
        print(f"  P(State_1 = '{state}') = {prob:.4f}")

except Exception as e:
    print("Error during inference.")
    print(f"Exception: {e}")

### Model evaluation and metrics

In [ ]:
tp = fp = fn = tn = 0

for _, r in df_transitions.iterrows():
    ev = {('State', 0): r['State']}
    for c in FEATURES:
        ev[(c, 0)] = r[c]

    true_state = r['State_next']

    try:
        q = dbn_infer.query(variables=[('State', 1)], evidence=ev, show_progress=False)

        fail_idx = q.state_names[('State', 1)].index('Failure')
        pred_state = 'Failure' if float(q.values[fail_idx]) >= 0.5 else 'Running'

    except Exception:
        pred_state = r['State']  # Fallback if network crashes on unseen data

    # 4. Count results
    tp += (true_state == 'Failure' and pred_state == 'Failure')
    fp += (true_state == 'Running' and pred_state == 'Failure')
    fn += (true_state == 'Failure' and pred_state == 'Running')
    tn += (true_state == 'Running' and pred_state == 'Running')


total = tp + fp + fn + tn
print(f"Total Evaluated: {total}")
print(f"TP: {tp} | FP: {fp}")
print(f"TN: {tn} | FN: {fn}")
print(f"Accuracy: {(tp + tn) / total:.2%}")